<a href="https://colab.research.google.com/github/Ranjitkumarmuduli/crop-disease-detection/blob/main/notebooks_01_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install kagglehub
import kagglehub, os
path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
   for root, dirs, files in os.walk(path):
       if os.path.basename(root) == "color":
           print(root, "|", len(dirs), "classes")
           for d in sorted(dirs):
               print(f"{len(os.listdir(os.path.join(root, d))):5d}  {d}")
           break

Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
/kaggle/input/plantvillage-dataset/plantvillage dataset/color | 38 classes
  630  Apple___Apple_scab
  621  Apple___Black_rot
  275  Apple___Cedar_apple_rust
 1645  Apple___healthy
 1502  Blueberry___healthy
 1052  Cherry_(including_sour)___Powdery_mildew
  854  Cherry_(including_sour)___healthy
  513  Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
 1192  Corn_(maize)___Common_rust_
  985  Corn_(maize)___Northern_Leaf_Blight
 1162  Corn_(maize)___healthy
 1180  Grape___Black_rot
 1383  Grape___Esca_(Black_Measles)
 1076  Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
  423  Grape___healthy
 5507  Orange___Haunglongbing_(Citrus_greening)
 2297  Peach___Bacterial_spot
  360  Peach___healthy
  997  Pepper,_bell___Bacterial_spot
 1478  Pepper,_bell___healthy
 1000  Potato___Early_blight
 1000  Potato___Late_blight
  152  Potato___healthy
  371  Raspberry___healthy
 5090  Soybean___healthy
 1835  Squash___Powdery_mil

In [4]:
import os, hashlib, pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive
drive.mount('/content/drive')

ROOT = "/kaggle/input/plantvillage-dataset/plantvillage dataset/color"
OUT  = "/content/drive/MyDrive/crop_disease_project/splits"
os.makedirs(OUT, exist_ok=True)

classes = sorted(os.listdir(ROOT))
df = pd.DataFrame([{"relpath": f"{c}/{f}", "label": i, "class": c}
                   for i, c in enumerate(classes) for f in sorted(os.listdir(f"{ROOT}/{c}"))])

# Exact duplicate check (identical file bytes)
df["md5"] = [hashlib.md5(open(f"{ROOT}/{p}", "rb").read()).hexdigest() for p in df.relpath]
conflict = df.groupby("md5")["label"].nunique()
conflict = conflict[conflict > 1].index
print("Duplicate images:", df.duplicated("md5").sum(), "| duplicates with different labels:", len(conflict))
df = df[~df.md5.isin(conflict)].drop_duplicates("md5").reset_index(drop=True)

# Stratified 70 / 15 / 15 split, fixed seed for reproducibility
train, temp = train_test_split(df, test_size=0.30, stratify=df.label, random_state=42)
val, test   = train_test_split(temp, test_size=0.50, stratify=temp.label, random_state=42)
for name, d in [("train", train), ("val", val), ("test", test)]:
    d[["relpath", "label", "class"]].to_csv(f"{OUT}/{name}.csv", index=False)
pd.Series(classes).to_csv(f"{OUT}/classes.csv", index=False, header=False)
print(f"Final: {len(df):,} images | train {len(train):,} | val {len(val):,} | test {len(test):,}")

Mounted at /content/drive
Duplicate images: 21 | duplicates with different labels: 0
Final: 54,284 images | train 37,998 | val 8,143 | test 8,143
